## Concept 3 — Advanced RAG (LLM Reranking)

### What is it?
Retrieval returns many candidate docs. Not all are equally useful.
Reranking adds a **second filtering stage** — an LLM scores each doc for relevance and only the top-scored docs go to the final LLM.

### Pipeline
```
Query
  → Retriever (get more docs, e.g. k=6)
  → Reranker: LLM scores each doc 1-10
  → Keep top 3 by score
  → LLM → Answer
```

### Why rerank?
Vector similarity is geometric distance — a doc can be 'close' in embedding space but not actually answer the question.
The LLM reranker understands meaning and can score true relevance.

### Limitation (what Concept 4 fixes)
Only vector search is used — misses exact keyword matches.
→ Concept 4 adds BM25 keyword search alongside vector search (Hybrid RAG).

In [2]:
print("Hi")

Hi


In [3]:
import sys
sys.path.insert(0, '.')

from RAGutils import setup, format_docs, TEST_QUERIES, run_queries

chunks, vectorstore, embeddings, llm, prompt = setup()

USER_AGENT environment variable not set, consider setting it to identify your requests.


Loading documents...
Splitting into chunks...
Creating embeddings and vector store...

Ready: 1 page(s) -> 11 chunks -> vector store created
LLM: gpt-4o-mini | Embeddings: text-embedding-3-small


### Step 1 — Retriever (Fetch More Candidates)
Fetch more docs than needed (k=6) so the reranker has enough to choose from.

In [4]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 6})

### Step 2 — LLM Reranker
Ask the LLM to score each doc against the query from 1-10. Sort and keep only top_k.

In [5]:
def rerank_docs(query, docs, top_k=3):
    scored = []
    for doc in docs:
        score_prompt = (
            f"Query: {query}\n"
            f"Document: {doc.page_content[:600]}\n\n"
            f"How relevant is this document to the query? Score 1-10 (return only the number):"
        )
        score_str = llm.invoke(score_prompt).content.strip()
        try:
            score = int(score_str)
        except ValueError:
            score = 0
        scored.append((score, doc))
        print(f"  Score {score}: {doc.page_content[:80]}...")

    scored.sort(reverse=True, key=lambda x: x[0])
    return [doc for _, doc in scored[:top_k]]

### Step 3 — See Reranking in Action
Watch how the LLM scores each doc and which ones get selected.

In [6]:
query = TEST_QUERIES[0]
print(f"Query: {query}\n")
print("Scoring docs...")

candidate_docs = retriever.invoke(query)
top_docs = rerank_docs(query, candidate_docs, top_k=3)

print(f"\nSelected top {len(top_docs)} docs after reranking")

Query: What is LangSmith persistence?

Scoring docs...
  Score 5: Core resource data: assistants, threads, runs, and cron jobs. Always stored in P...
  Score 1: Agent Server - Docs by LangChainDocumentation IndexFetch the complete documentat...
  Score 3: Use Agent Server to create and manage:
AssistantsThreadsRunsCron jobs
API refere...
  Score 3: built-in backendsExtend the HTTP serverHeaders and loggingOn this pageApplicatio...
  Score 1: Compiled graph (recommended): Export an already-compiled CompiledGraph instance....
  Score 1: ​Graphs
When you deploy a graph with Agent Server, you are deploying a “blueprin...

Selected top 3 docs after reranking


### Step 4 — Full Pipeline Function

In [7]:
def advanced_rag(query):
    candidate_docs = retriever.invoke(query)
    top_docs       = rerank_docs(query, candidate_docs, top_k=3)
    context        = format_docs(top_docs)
    response       = llm.invoke(prompt.format_messages(context=context, question=query))
    return response.content

### Test — Standard Queries

In [8]:
run_queries(advanced_rag, label="Concept 3 — Advanced RAG (Reranking)")


 Concept 3 — Advanced RAG (Reranking) — Test Results

Q: What is LangSmith persistence?
  Score 2: Core resource data: assistants, threads, runs, and cron jobs. Always stored in P...
  Score 1: Agent Server - Docs by LangChainDocumentation IndexFetch the complete documentat...
  Score 2: Use Agent Server to create and manage:
AssistantsThreadsRunsCron jobs
API refere...
  Score 4: built-in backendsExtend the HTTP serverHeaders and loggingOn this pageApplicatio...
  Score 1: Compiled graph (recommended): Export an already-compiled CompiledGraph instance....
  Score 1: ​Graphs
When you deploy a graph with Agent Server, you are deploying a “blueprin...
A: LangSmith persistence consists of core resource data (assistants, threads, runs, and cron jobs) that are always stored in PostgreSQL. It includes two types of memory: checkpoints (short-term memory) that are snapshots of graph execution state, and a store (long-term memory) that persists information across threads. The default storage i